# Analyze and control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/basics/02_analyze_and_control.ipynb)

Linearize a model, read its poles, and design LQR, PID and LQG controllers, with Bode, margins and root locus.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. Linearize at an operating point and read the poles

Near an operating point $(\bar x, \bar u)$ a nonlinear model behaves like its
linearization,

$$\delta \dot x = A\, \delta x + B\, \delta u, \qquad
A = \frac{\partial f}{\partial x}\Big|_{\bar x, \bar u}, \quad
B = \frac{\partial f}{\partial u}\Big|_{\bar x, \bar u}.$$

`linearize` computes $A$, $B$, $C$, $D$ and returns them as a linear system. The pendulum's
angle is measured from the bottom, so $\bar x = [\pi, 0]$ is the upright position.

In [ ]:
import numpy as np

from minilink import Pendulum

pendulum = Pendulum()
x_up = np.array([np.pi, 0.0])
lin = pendulum.linearize(x_up)
lin

The poles are the eigenvalues of $A$. Upright, one is positive: the position is unstable.
Hanging down, the pair sits on the imaginary axis: an undamped oscillation.

In [ ]:
poles_up = np.linalg.eigvals(lin.A())
poles_down = np.linalg.eigvals(pendulum.linearize(np.zeros(2)).A())
print("upright:", poles_up)
print("down:   ", poles_down)

## 2. LQR stabilizes the operating point

`lqr` designs the state feedback $u = \bar u - K (x - \bar x)$ that minimizes
$\int \delta x^T Q\, \delta x + \delta u^T R\, \delta u \; dt$ on that linearization, and
returns a controller ready for `@`. Started 0.5 rad from the top, the pendulum swings up
and stays there.

In [ ]:
from minilink import lqr

Q = np.diag([10.0, 1.0])
R = np.array([[1.0]])
lqr_controller = lqr(pendulum, Q, R, x_bar=x_up)
lqr_controller.params["K"]

In [ ]:
pendulum.x0 = np.array([np.pi - 0.5, 0.0])
lqr_loop = lqr_controller @ pendulum
lqr_loop.compute_trajectory(tf=5.0)
lqr_loop.plot_trajectory()

## 3. The PID family

`P`, `PI`, `PD` and `PID` act on the error $e = r - y$; `@` inserts the junction.
`Impedance` reads the measured rate instead of filtering the derivative of $y$. Each law
holds a damped pendulum at $r = 0.5$ rad against gravity. The integral term removes the
static error.

In [ ]:
import matplotlib.pyplot as plt

from minilink import PD, PI, PID, Impedance, P, Step, closed_loop

plant = Pendulum()
plant.params["d"] = 2.0
reference = Step(final_value=[0.5], step_time=1.0)

family = {
    "P": P(Kp=20.0),
    "PI": PI(Kp=20.0, Ki=10.0),
    "PD": PD(Kp=20.0, Kd=2.0, tau=0.05),
    "PID": PID(Kp=20.0, Ki=10.0, Kd=2.0, tau=0.05),
    "Impedance": Impedance(Kp=20.0, Kd=2.0),
}

for name, controller in family.items():
    loop = closed_loop(controller, plant, reference=reference)
    traj = loop.compute_trajectory(tf=10.0, verbose=False)
    plt.plot(traj.t, traj.of("plant").x[0], label=name)  # the plant's angle

plt.axhline(0.5, color="k", linestyle=":")
plt.xlabel("t [s]")
plt.ylabel("theta [rad]")
plt.legend()
plt.show()

## 4. Frequency response: Bode, margins and root locus

`C >> G` is the series connection, the loop gain $L = C G$ before the loop closes. Its
`margins` say how much gain and phase the closed loop can lose before it goes unstable;
`plot_bode` draws them on the Bode diagram.

In [ ]:
from minilink import margins

C = PID(Kp=20.0, Ki=10.0, Kd=2.0, tau=0.05)
L = C >> plant
margins(L)

In [ ]:
L.plot_bode()

The root locus follows the closed-loop poles of $k L$ as the gain $k$ grows from zero.

In [ ]:
L.plot_root_locus()

## 5. Observer and LQG

LQR reads the whole state. A real sensor often gives less: here a cart-pole measures its
two positions, not the velocities. `kalman` designs the observer that rebuilds $x$ from
$u$ and $y$, and `lqg` puts the LQR gain on its estimate: one controller with `y` in and
`u` out, closed with `@`.

In [ ]:
from minilink import CartPole, Gain, kalman, lqg

cartpole = CartPole()
cartpole.x0 = np.array([-1.0, np.pi - 0.2, 0.0, 0.0])
sensed = cartpole >> Gain(np.eye(4)[:2])  # measure the cart and pole positions only

x_up = np.array([0.0, np.pi, 0.0, 0.0])
Q = np.eye(4)
R = np.array([[1.0]])
Q_n = 1e-2 * np.eye(4)  # process-noise intensity
R_n = 1e-4 * np.eye(2)  # measurement-noise intensity

kalman(sensed, Q_n, R_n, x_bar=x_up)

In [ ]:
lqg_loop = lqg(sensed, Q, R, Q_n, R_n, x_bar=x_up) @ sensed
lqg_loop.compute_trajectory(tf=8.0)
lqg_loop.plot_trajectory()

## 6. Blocks, `>>` and `feedback`

Every block is a system, and `>>` chains them: sources (`Step`, `Sine`, `WhiteNoise`),
gains and sums, filters, nonlinearities and transfer functions. A second-order
`TransferFunction`, $G(s) = \omega_n^2 / (s^2 + 2 \zeta \omega_n s + \omega_n^2)$, driven
by a step:

In [ ]:
from minilink import TransferFunction

wn, zeta = 2.0, 0.3
G = TransferFunction([wn**2], [1.0, 2 * zeta * wn, wn**2])
step_response = Step(final_value=1.0, step_time=1.0) >> G
step_response.compute_trajectory(tf=10.0)
step_response.plot_trajectory()

`feedback` closes a chain on itself with unity negative feedback. Here a `Saturation` limits the PID's command on a double
integrator. The integral term keeps growing while the command is clipped (windup), so the
response overshoots far and oscillates.

In [ ]:
from minilink import DoubleIntegrator, Saturation, feedback

open_loop = PID(Kp=5.0, Ki=1.0, Kd=3.0) >> Saturation(-0.5, 0.5) >> DoubleIntegrator()
windup_loop = Step(final_value=4.0, step_time=1.0) >> feedback(open_loop)
windup_loop.compute_trajectory(tf=20.0)
windup_loop.plot_trajectory()

The fix is to give the limits to the PID itself: with `u_min` and `u_max` it clips its own
command and stops integrating while saturated (anti-windup).

In [ ]:
pid = PID(Kp=5.0, Ki=1.0, Kd=3.0, u_min=-0.5, u_max=0.5)
anti_windup_loop = Step(final_value=4.0, step_time=1.0) >> pid @ DoubleIntegrator()
anti_windup_loop.compute_trajectory(tf=20.0)
anti_windup_loop.plot_trajectory()

`closed_loop` also opens a load-disturbance port `d`, added to the command, and a
measurement-noise port `n`, added to the measurement. A `WhiteNoise` source drives each
one. White noise is drawn on a time grid, so the simulation uses a fixed-step solver.

In [ ]:
from minilink import WhiteNoise

noisy_loop = closed_loop(
    Impedance(Kp=50.0, Kd=10.0),
    Pendulum(),
    reference=Step(final_value=[1.0], step_time=1.0),
    disturbance=WhiteNoise(1, psd=0.1, seed=1),
    noise=WhiteNoise(2, psd=1e-4, seed=2),
)
noisy_loop.compute_trajectory(tf=10.0, solver="euler", dt=0.01)
noisy_loop.plot_trajectory()

## Where to go next

- [03_plan_and_sample](03_plan_and_sample.ipynb): pose a planning problem, compare a
  planner against a controller, and run a controller on a digital clock.
- [core/](../core/01_systems.ipynb): the objects underneath these shortcuts.